# TGR2026 Track 2: เทรน YOLO26 บนโน้ตบุ๊ก
**ออกแบบเนื้อหาโดย Dr. Natthakorn Kasamsumran**

ลำดับ: ตรวจเครื่อง → ตัดเมล็ดอัตโนมัติ → สร้างกองสังเคราะห์ → เทรน cls (วันที่ 1) → เทรน seg (วันที่ 2–3) → ทดสอบกับภาพกอง → export NCNN → ส่งขึ้น Pi

รันทีละเซลล์ด้วย **Shift+Enter** ใช้ kernel **Python (tgr)**

In [ ]:
# 1) ตรวจเครื่อง: ต้องเห็น CUDA True และชื่อ GPU
import platform, torch, ultralytics
print(platform.platform())
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(p.name, round(p.total_memory / 1e9, 1), 'GB')
print('ultralytics', ultralytics.__version__)

In [ ]:
# 2) ตั้งค่าเส้นทาง (แก้ให้ตรงกับข้อมูลที่ได้รับ)
from pathlib import Path
import os
ROOT = Path.home() / 'tgr_vision'          # โฟลเดอร์ที่มีสคริปต์ทั้งหมด
RAW = ROOT / 'data/day1_raw'               # data/day1_raw/<class>/*.jpg  (1 ภาพ = 1 คลาส)
TEST = ROOT / 'data/day1_test'             # ภาพกองทดสอบ (ถ้ามี)
EXT = ROOT / 'data/day1_out'               # ผลของ bean_extract.py
SYN = ROOT / 'data/synth'                  # ผลของ synth_piles.py
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
IMGSZ_SEG = 960                            # ขนาดภาพของ seg (ต้องใช้ค่าเดียวกันตอน export และบน Pi)
os.chdir(ROOT)
print('classes:', sorted(p.name for p in RAW.iterdir() if p.is_dir()))

In [ ]:
# 3) ตัดเมล็ดอัตโนมัติ (ได้ crop สำหรับ cls + polygon สำหรับ seg + cutout สำหรับกองสังเคราะห์)
!python bean_extract.py --src {RAW} --out {EXT}

In [ ]:
# 4) ตรวจภาพ debug ทุกภาพ: กรอบเขียวต้องครอบทุกเมล็ด ไม่มีกรอบบนพื้นไม้
import cv2, matplotlib.pyplot as plt
dbg = sorted((EXT / 'debug').glob('*.jpg'))
cols = 3; rows = (len(dbg) + cols - 1) // cols
plt.figure(figsize=(18, 5 * rows))
for i, p in enumerate(dbg):
    plt.subplot(rows, cols, i + 1); plt.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)); plt.title(p.name); plt.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 5) สร้างกองเมล็ดสังเคราะห์ (ทั้งแบบคลาสเดียวและแบบคละ)
!python synth_piles.py --ext {EXT} --out {SYN} --n 400 --mode both

In [ ]:
# 6) ตรวจ label ของกองสังเคราะห์ด้วยตา
import numpy as np, random
def show_label(img_path):
    im = cv2.imread(str(img_path)); H, W = im.shape[:2]
    lab = Path(str(img_path).replace('/images/', '/labels/')).with_suffix('.txt')
    for ln in lab.read_text().split('\n'):
        if ln.strip():
            v = ln.split(); pts = (np.array(v[1:], float).reshape(-1, 2) * [W, H]).astype(np.int32)
            cv2.polylines(im, [pts], True, (0, 255, 0), 2)
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
samples = random.sample(sorted((SYN / 'images/train').glob('*.jpg')), 4)
plt.figure(figsize=(18, 12))
for i, p in enumerate(samples):
    plt.subplot(2, 2, i + 1); plt.imshow(show_label(p)); plt.axis('off')
plt.show()

In [ ]:
# 7) วันที่ 1: เทรน YOLO26n-cls กับภาพเมล็ดทีละเม็ด
from ultralytics import YOLO
cls = YOLO('yolo26n-cls.pt')
cls.train(data=str(EXT / 'cls'), epochs=60, imgsz=224, batch=64, device=DEVICE, patience=20,
          fliplr=0.5, flipud=0.5, hsv_h=0.01, hsv_s=0.3, hsv_v=0.3,
          project=str(ROOT / 'runs'), name='day1_cls', exist_ok=True)

In [ ]:
# 8) วันที่ 2–3: เทรน YOLO26n-seg กับกองสังเคราะห์ + ภาพจริง
# batch=-1 ให้ Ultralytics เลือกขนาด batch ตามหน่วยความจำ GPU เอง (GTX 1650 4GB มักได้ 4–8)
seg = YOLO('yolo26n-seg.pt')
seg.train(data=str(SYN / 'data.yaml'), epochs=120, imgsz=IMGSZ_SEG, batch=-1, device=DEVICE, patience=30,
          degrees=180, fliplr=0.5, flipud=0.5, scale=0.3, mosaic=1.0, close_mosaic=15,
          hsv_h=0.01, hsv_s=0.4, hsv_v=0.3,
          project=str(ROOT / 'runs'), name='day2_seg', exist_ok=True)

In [ ]:
# 9) ดู confusion matrix และกราฟการเทรน
from IPython.display import Image, display
for f in ['day1_cls/confusion_matrix_normalized.png', 'day2_seg/confusion_matrix_normalized.png', 'day2_seg/results.png']:
    p = ROOT / 'runs' / f
    if p.exists():
        print(f); display(Image(filename=str(p), width=900))

In [ ]:
# 10) ทดสอบกับภาพกองจริง: นับเมล็ดต่อคลาส แล้วโหวตเป็นคลาสของภาพ
!python predict_pi.py --model runs/day2_seg/weights/best.pt --imgsz {IMGSZ_SEG} --source {TEST} --save --out runs_test
import pandas as pd
pd.read_csv('runs_test/summary.csv')

In [ ]:
# 11) (ทางเลือก) สองขั้น: seg หาตำแหน่งเมล็ด แล้ว cls จำแนกทีละเม็ด เทียบความแม่นกับข้อ 10
!python predict_pi.py --model runs/day2_seg/weights/best.pt --cls-model runs/day1_cls/weights/best.pt --imgsz {IMGSZ_SEG} --source {TEST} --out runs_test_2stage
pd.read_csv('runs_test_2stage/summary.csv')

In [ ]:
# 12) export เป็น NCNN แล้วรวมไฟล์ส่งขึ้น Pi
import shutil
VER = 'v1'                                   # เปลี่ยนทุกครั้งที่เทรนใหม่ จะได้รู้ว่า Pi ใช้โมเดลไหน
YOLO('runs/day1_cls/weights/best.pt').export(format='ncnn', imgsz=224)
YOLO('runs/day2_seg/weights/best.pt').export(format='ncnn', imgsz=IMGSZ_SEG)
dep = ROOT / 'deploy'; dep.mkdir(exist_ok=True)
for src, dst in [('runs/day1_cls/weights/best_ncnn_model', f'cls_{VER}_ncnn_model'),
                 ('runs/day2_seg/weights/best_ncnn_model', f'seg_{VER}_{IMGSZ_SEG}_ncnn_model')]:
    shutil.rmtree(dep / dst, ignore_errors=True); shutil.copytree(src, dep / dst)
print(sorted(p.name for p in dep.iterdir()))

In [ ]:
# 13) ส่งโมเดลและสคริปต์ขึ้น Pi (แก้ชื่อเครื่องให้ตรง)
!scp -r deploy/*_ncnn_model *.py pi@tgr-vision.local:~/tgr_vision/